# Animation Engine — selected humanoid → rig → ARDY → animated FBX

This engine is **independent** from the 3D Engine.

Use it only when you manually choose a humanoid character that should be animated.
Props, furniture, buildings, etc. never need to enter this notebook.

Pipeline:

`selected humanoid GLB/FBX → Make-It-Animatable rig → ARDY text motion → FBX bridge → retarget → animated FBX`

**Important:** ARDY officially outputs `.npz` motion. The FBX/retarget bridge in this repository is integration glue and should be treated as experimental until validated on your character.

In [ ]:
!nvidia-smi

In [ ]:
# Common tools + this repository.
import os, pathlib, shutil

!apt-get update -qq
!apt-get install -y -qq git git-lfs build-essential cmake ninja-build wget ffmpeg

if not pathlib.Path("/opt/conda/bin/conda").exists():
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /opt/conda

if pathlib.Path("/content/My-works").exists():
    shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works

TOOLS = "/content/My-works/ai-3d-animation-engines/animation-engine"
OUTPUT_DIR = "/content/animation_outputs"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Helpers:", TOOLS)

## 1. Install ARDY in its own environment

ARDY currently requires PyTorch 2.4+ and uses a gated Llama-3-8B-Instruct text encoder.
Keeping it in a separate Conda environment avoids fighting Make-It-Animatable's older PyTorch stack.

In [ ]:
%%bash
set -euo pipefail
source /opt/conda/etc/profile.d/conda.sh

rm -rf /content/ardy
git clone -q https://github.com/nv-tlabs/ardy.git /content/ardy

conda env remove -n ardy -y >/dev/null 2>&1 || true
conda create -n ardy python=3.11 -y -q

conda run -n ardy python -m pip install -q --upgrade pip
conda run -n ardy python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

cd /content/ardy
conda run -n ardy python -m pip install -e ".[demo]"
echo "ARDY installed."

## 2. Hugging Face access for ARDY text prompts

Before running this cell, your Hugging Face account must have access to
`meta-llama/Meta-Llama-3-8B-Instruct`.

The token is requested interactively and is **not written to this notebook**.

In [ ]:
import os, subprocess, getpass

hf_token = getpass.getpass("Hugging Face token: ")
os.environ["HF_TOKEN"] = hf_token

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "ardy",
    "python", "-c",
    "import os; from huggingface_hub import login; login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)"
], check=True)

print("Hugging Face authentication stored only in this Colab runtime.")

## 3. Install Make-It-Animatable in a separate environment

This gives the animation engine a Mixamo-style humanoid rig and the bundled retargeting path.

In [ ]:
%%bash
set -euo pipefail
source /opt/conda/etc/profile.d/conda.sh

rm -rf /content/Make-It-Animatable /tmp/mia-hf-data
git clone -q https://github.com/jasongzy/Make-It-Animatable /content/Make-It-Animatable --recursive --single-branch

conda env remove -n mia -y >/dev/null 2>&1 || true
conda create -n mia python=3.11 -y -q

cd /content/Make-It-Animatable
conda run -n mia python -m pip install -q --upgrade pip
conda run -n mia python -m pip install -r requirements.txt

git lfs install --skip-repo >/dev/null

# Minimal runtime data needed by the current upstream inference code.
mkdir -p data
GIT_LFS_SKIP_SMUDGE=1 git -C data clone -q https://huggingface.co/datasets/jasongzy/Mixamo
GIT_LFS_SKIP_SMUDGE=1 git clone -q https://huggingface.co/jasongzy/Make-It-Animatable /tmp/mia-hf-data

git -C data/Mixamo lfs pull -I 'bones*.fbx'
git -C /tmp/mia-hf-data lfs pull -I 'output/best/new'

mkdir -p output/best
cp -r /tmp/mia-hf-data/output/best/new output/best/

wget -q https://github.com/facebookincubator/FBX2glTF/releases/download/v0.9.7/FBX2glTF-linux-x64 -O util/FBX2glTF
chmod +x util/FBX2glTF

echo "Make-It-Animatable installed."

## 4. Manually select the target character

Upload the humanoid generated by TRELLIS.2 (or any other humanoid GLB/FBX).
This explicit selection is the boundary between the two engines.

In [ ]:
from google.colab import files
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one humanoid .glb or .fbx.")

target_name = next(iter(uploaded))
TARGET_CHARACTER = f"/content/{target_name}"
if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {".glb", ".fbx", ".obj", ".ply"}:
    raise ValueError("Expected GLB/FBX/OBJ/PLY.")

print("Selected target:", TARGET_CHARACTER)

## 5. Generate motion with ARDY

In [ ]:
import subprocess, shlex, os

PROMPT = "A person walks forward, stops, and waves with the right hand."
DURATION_SECONDS = 6.0
SEED = 0

motion_stem = f"{OUTPUT_DIR}/motion"

cmd = [
    "/opt/conda/bin/conda", "run", "-n", "ardy",
    "python", "scripts/generate.py",
    PROMPT,
    "--model", "core",
    "--duration", str(DURATION_SECONDS),
    "--seed", str(SEED),
    "--output", motion_stem,
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/ardy", env=os.environ.copy(), check=True)

MOTION_NPZ = f"{motion_stem}.npz"
print("ARDY motion:", MOTION_NPZ)

In [ ]:
# Add Core skeleton names, hierarchy and neutral joints for downstream FBX conversion.
MOTION_BRIDGE = f"{OUTPUT_DIR}/motion_bridge.npz"

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "ardy",
    "python", f"{TOOLS}/enrich_ardy_motion.py",
    "--input", MOTION_NPZ,
    "--output", MOTION_BRIDGE,
], check=True)

print("Bridge motion:", MOTION_BRIDGE)

In [ ]:
# Quick skeleton preview in the notebook.
MOTION_PREVIEW = f"{OUTPUT_DIR}/motion_preview.mp4"

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "ardy",
    "python", f"{TOOLS}/preview_ardy_motion.py",
    "--input", MOTION_BRIDGE,
    "--output", MOTION_PREVIEW,
], check=True)

from IPython.display import Video, display
display(Video(MOTION_PREVIEW, embed=True))

## 6. Convert ARDY motion into an FBX source skeleton

This does **not** touch the selected target yet. It simply turns ARDY's generated motion into an animation source that the retargeter can consume.

In [ ]:
ARDY_SOURCE_FBX = f"{OUTPUT_DIR}/ardy_source.fbx"

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "mia",
    "python", f"{TOOLS}/ardy_motion_to_fbx.py",
    "--input", MOTION_BRIDGE,
    "--output", ARDY_SOURCE_FBX,
], check=True)

print("ARDY source FBX:", ARDY_SOURCE_FBX)

## 7. Auto-rig the manually selected character

If you upload a character that is already correctly rigged to a Mixamo-compatible skeleton,
set `TARGET_ALREADY_RIGGED = True` and provide that FBX directly.

In [ ]:
import pathlib

TARGET_ALREADY_RIGGED = False
RIGGED_TARGET = f"{OUTPUT_DIR}/character_rigged.fbx"

if TARGET_ALREADY_RIGGED:
    if pathlib.Path(TARGET_CHARACTER).suffix.lower() != ".fbx":
        raise ValueError("For skip-rig mode, upload a rigged FBX.")
    shutil.copy2(TARGET_CHARACTER, RIGGED_TARGET)
else:
    subprocess.run([
        "/opt/conda/bin/conda", "run", "-n", "mia",
        "python", f"{TOOLS}/rig_character_mia.py",
        "--input", TARGET_CHARACTER,
        "--output", RIGGED_TARGET,
        "--no-fingers",
    ], env={**os.environ, "MIA_ROOT": "/content/Make-It-Animatable"}, check=True)

print("Rigged target:", RIGGED_TARGET)

## 8. Retarget ARDY motion to the selected character

The final FBX is the handoff to Unreal Engine.

In [ ]:
FINAL_FBX = f"{OUTPUT_DIR}/character_animated.fbx"
FINAL_GLB = f"{OUTPUT_DIR}/character_animated.glb"

subprocess.run([
    "/opt/conda/bin/conda", "run", "-n", "mia",
    "python", f"{TOOLS}/retarget_with_mia.py",
    "--target", RIGGED_TARGET,
    "--animation", ARDY_SOURCE_FBX,
    "--output", FINAL_FBX,
    "--preview-glb", FINAL_GLB,
], env={**os.environ, "MIA_ROOT": "/content/Make-It-Animatable"}, check=True)

print("Final Unreal handoff:", FINAL_FBX)
print("Optional GLB preview:", FINAL_GLB)

In [ ]:
# Download the main outputs.
from google.colab import files
files.download(FINAL_FBX)

## Optional — save the whole animation job to Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/AI-Animation-Engine"
# !cp -f /content/animation_outputs/* "/content/drive/MyDrive/AI-Animation-Engine/"